# 02 · 3-D blade — C2 NURBS skin, solid, hub fillet

A blade is a set of **stations** (span, profile, placement) on stream surfaces of a **meridional channel**.
Between stations every profile parameter is interpolated in span (natural cubic splines), each section is
placed in the conformal plane (m′, θ) of its stream line (angles are the true flow angles) and mapped to 3-D
(x axial, y = r sin θ, z = r cos θ).

OpenCASCADE then builds:

1. the **skin**: every section interpolated by a *periodic C2* cubic B-spline with common parameters, the
   poles interpolated spanwise (Piegl & Tiller skinning) → one B-spline surface, periodic around the profile,
   C2 in both directions. (OCCT's own periodic interpolation puts a double knot at the seam — only C1 —
   so BladeGen solves the periodic system itself.)
2. the **solid**: the skin, extended past hub and casing, splits an annulus sector → caps exactly on the hub
   and casing surfaces of revolution;
3. on request the **hub fillet**: a rolling-ball fillet built face by face (OCCT's BRepFilletAPI fails on
   this junction).

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# the bladegen package lives in <build>/python (cmake --preset python); BLADEGEN_PY overrides
REPO = Path.cwd().resolve().parents[1]
for cand in [os.environ.get("BLADEGEN_PY", ""), REPO / "build" / "python" / "python"]:
    if cand and Path(cand).is_dir() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))
import bladegen as bg
import bladegen.notebook as nb
OUT = Path(os.environ.get("BLADEGEN_TEST_OUTPUT", REPO / "build" / "notebook_output"))
OUT.mkdir(parents=True, exist_ok=True)
print("BladeGen", bg.__version__, "| widgets:", nb.HAVE_WIDGETS)

In [ ]:
blade = bg.Blade(bg.BladeDefinition.demo_rotor())
secs = blade.sections(30, 100)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
nb.meridional_view(ax[0], blade, secs); ax[0].set_title("meridional view: channel and sections")
for s in secs:
    ax[1].plot(*s["conformal"].T, lw=1, color=plt.cm.viridis(min(max(s["span"], 0), 1)))
ax[1].set_aspect("equal"); ax[1].grid(alpha=.3); ax[1].set_xlabel("m'"); ax[1].set_ylabel("θ [rad]")
ax[1].set_title("sections in their conformal planes (colour = span)")
plt.show()

## Station editor

Pick a station; its profile knobs (grouped by family) and placement knobs change only that station. The
plot shows the section at that span with its quality badge and all sections in 3-D.

In [ ]:
definition = bg.BladeDefinition.demo_rotor()

def station_view(i=1):
    st = definition.stations[i]
    fig, prof, q = nb.profile_dashboard(st.profile, title=f"station {i} (span {st.span:.2f})")
    plt.show()

if nb.HAVE_WIDGETS:
    import ipywidgets as W
    from IPython.display import display
    pick = W.Dropdown(options=[(f"station {i}: span {s.span:.2f}", i) for i, s in enumerate(definition.stations)],
                      value=1, description="station")
    pl = {k: W.FloatSlider(value=getattr(definition.stations[1].placement, k), min=lo, max=hi, step=(hi - lo) / 400,
                           description=k, continuous_update=False)
          for k, lo, hi in [("stagger_deg", -10, 75), ("m_le", 0.05, 0.6), ("m_te", 0.4, 0.95), ("theta_le_deg", -10, 10)]}
    out = W.Output()
    knobs = nb.ProfileKnobs(definition.stations[1].profile)
    def refresh(*_):
        i = pick.value
        st = definition.stations[i]
        st.profile = knobs.params.copy()
        p = st.placement
        for k, s in pl.items():
            setattr(p, k, s.value)
        st.placement = p
        stations = list(definition.stations); stations[i] = st; definition.stations = stations
        with out:
            out.clear_output(wait=True)
            b = bg.Blade(definition)
            fig = plt.figure(figsize=(15, 5))
            a = fig.add_subplot(1, 2, 1)
            sec = b.section(st.span, 30, 100)
            q = bg.check_quality(bg.Profile2D(st.profile))
            a.plot(*sec["conformal"].T, color="k"); a.set_aspect("equal"); a.grid(alpha=.3)
            a.set_title(f"station {i}: {q['summary'].splitlines()[0]}", color="red" if q["errors"] else "green")
            a3 = fig.add_subplot(1, 2, 2, projection="3d")
            for s in b.sections(20, 60):
                a3.plot(*(s["xyz"].T * 1000), color="steelblue", lw=0.8)
            a3.set_box_aspect((1, 1, 1)); a3.set_title("all sections [mm]")
            plt.show()
    def on_pick(change):
        st = definition.stations[pick.value]
        knobs.set_params(st.profile)
        for k, s in pl.items():
            s.value = getattr(st.placement, k)
    pick.observe(on_pick, names="value")
    knobs.observe(lambda p: refresh())
    for s in pl.values():
        s.observe(refresh, names="value")
    display(W.VBox([W.HBox([pick] + list(pl.values())), knobs.widget, out]))
    refresh()
else:
    station_view(1)

## The NURBS skin: degree, periodicity, C2 across the seam, curvature

`seam_jumps` evaluates the first and second u-derivatives on both sides of the periodic seam: relative
jumps ≈ 1e-10 mean C2. The mean-curvature map shows the surface fairness over (u, v).

In [ ]:
blade = bg.Blade(definition)
shapes = bg.build_blade(blade, 0.0, 40, 160)
S = shapes.surface
print(f"skin: degree ({S.u_degree}, {S.v_degree}), poles {S.n_poles}, u-periodic {S.u_periodic}, OCCT continuity {S.continuity}")
print(f"interpolation error at the section points: {shapes.skin_error:.2e} m")
for v in (0.1, 0.5, 0.9):
    gap, d1, d2 = S.seam_jumps(v)
    print(f"seam at v = {v}: gap {gap:.1e} m, |Δ d1| / |d1| = {d1:.1e}, |Δ d2| / |d2| = {d2:.1e}")
print(f"blade solid: valid {shapes.blade_valid}, volume {1e6 * shapes.blade_volume:.2f} cm³, area {1e4 * shapes.blade_area:.1f} cm²")
K, H = S.curvatures(240, 60)
fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(H.T, origin="lower", aspect="auto", extent=[0, 1, 0, 1], cmap="coolwarm",
               vmin=-np.nanpercentile(abs(H), 98), vmax=np.nanpercentile(abs(H), 98))
plt.colorbar(im, label="mean curvature [1/m]"); ax.set_xlabel("u (around the profile, from the LE point)"); ax.set_ylabel("v (span)")
ax.set_title("mean curvature of the C2 skin"); plt.show()

## 3-D view with the **Add hub fillet** button

The button builds the rolling-ball fillet (radius in mm) in C++: ball centres on the hub contour offset by r
and on the blade surface offset by r along its fluid-side normal, circular arcs between the two contact
curves, a periodic C2 fillet surface, and a valid B-rep with exact shared edges. **Export** writes STEP and STL
(mm) of the blade and of the hub + blade (+ fillet).

In [ ]:
state = nb.fillet_panel(bg.Blade(definition), radius_mm=3.0, out_dir=str(OUT), name="demo_rotor")
if not nb.HAVE_WIDGETS:
    fig = state if not isinstance(state, dict) else None
    if fig is not None:
        try:
            fig.show()
        except Exception:
            plt.show()